# Week 5 -- Function 4: PyTorch NN Surrogate + Bayesian Optimisation (4D)

In [ ]:
# -- WEEKLY OBSERVATIONS LOG
import numpy as np

INITIAL_SIZE = 30
DATA_PATH_X  = '../data/function_4/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_4/initial_outputs.npy'

weekly_log = [
    ([0.888889, 0.555556, 0.777778, 0.777778], -24.548024),  # W1: UCB beta=2.5 -- BEST
    ([1.0, 0.666667, 0.111111, 0.777778], -28.563568),  # W2: UCB beta=3.5 -- worsened
    ([1.0, 0.666667, 0.111111, 0.777778], -28.563568),  # W3: EI -- stuck same region
    ([1.0, 1.0, 1.0, 0.0], -48.00036259295127),  # W4: NN grad ascent -- -48 (boundary, worst)
    # Week 5: add result here as ([x...], y_value)
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

In [ ]:
# Cell 3: Load data and inspect
# Function 4: Warehouse Placement (4D), Maximise

X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 72)
print('  All observations (sorted descending by Y)')
print('=' * 72)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.6f}' for v in x_val)
    print(f'  [{i+1:2d}]  X = [{x_str}]   Y = {y_val:+.6e}{marker}')
print('=' * 72)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
print(f'\n  Best Y*  = {best_Y:.6e}')
best_x_str2 = ', '.join(f'{v:.8f}' for v in best_X)
print(f'  Best X*  = [{best_x_str2}]')

In [ ]:
# Cell 4: Fit GP (kept as fallback / comparison)
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

Y_fit = np.log(np.abs(Y) + 1e-300)
kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
gp.fit(X, Y_fit)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                  : {gp.kernel_}')
print(f'  Log-marginal-likelihood : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
actual_log = np.log(np.abs(best_Y) + 1e-300)
print(f'  Sanity check at best X* : GP mean={pred_mean[0]:.4f}  actual={actual_log:.4f}  std={pred_std[0]:.6f}')
print('=' * 62)

In [ ]:
# Cell 5: PyTorch NN Surrogate (Assignment 16.1 style)
# Backpropagation via loss.backward() and Adam optimizer.

import torch
import torch.nn as nn
from torch.autograd import Variable
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

# Scale Y directly -- preserves ordering for maximisation across all functions
y_scaler = StandardScaler()
Y_scaled = y_scaler.fit_transform(Y.reshape(-1, 1)).ravel().astype(np.float32)

X_t = torch.tensor(X.astype(np.float32))
Y_t = torch.tensor(Y_scaled)

# nn.Module surrogate (Assignment 16.1 pattern)
class SurrogateNN(nn.Module):
    def __init__(self, n_in, h1=64, h2=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, h1),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(h2, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

model = SurrogateNN(n_dims)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)

# Training loop -- loss.backward() follows Assignment 16.1 backpropagation
for epoch in range(1500):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_t)
    loss = loss_fn(y_pred, Y_t)
    loss.backward()   # backpropagation (Assignment 16.1)
    optimizer.step()
    if epoch % 300 == 0:
        print(f'  Epoch {epoch:4d}  MSE (scaled) = {loss.item():.4f}')

print(f'\n  Final MSE (scaled): {loss.item():.4f}')
model.eval()
with torch.no_grad():
    nn_pred_best = model(torch.tensor(best_X.astype(np.float32))).item()
print(f'  NN prediction at best X*: {nn_pred_best:.4f} (scaled)')

In [ ]:
# Cell 6: Gradient-Guided Acquisition (PyTorch, Assignment 16.1 style)
# Strategy A: gradient ascent on input with requires_grad + backward
# Strategy B: GP EI on 30k random grid
# Winner: higher NN-predicted score

trust_radius = 0.1
lo = np.clip(best_X - trust_radius, 0.0, 1.0).astype(np.float32)
hi = np.clip(best_X + trust_radius, 0.0, 1.0).astype(np.float32)
lo_t = torch.tensor(lo)
hi_t = torch.tensor(hi)

# Strategy A: NN gradient ascent (Assignment 16.1: requires_grad input, backward)
x_opt = torch.tensor(best_X.copy().astype(np.float32), requires_grad=True)
opt_inner = torch.optim.Adam([x_opt], lr=5e-3)

model.eval()
for step in range(800):
    opt_inner.zero_grad()
    score = model(x_opt)
    (-score).backward()   # maximise score via backward (Assignment 16.1)
    opt_inner.step()
    with torch.no_grad():
        x_opt.clamp_(lo_t, hi_t)

grad_candidate = x_opt.detach().numpy().copy()

# GP EI on random grid
from scipy.stats import norm
best_log_y = np.log(np.abs(best_Y) + 1e-300)
np.random.seed(42)
X_grid = np.random.uniform(0, 1, size=(30000, n_dims))
gp_mean, gp_std = gp.predict(X_grid, return_std=True)
improvement = gp_mean - best_log_y
Z = improvement / (gp_std + 1e-9)
ei_scores = improvement * norm.cdf(Z) + gp_std * norm.pdf(Z)
gp_candidate = X_grid[np.argmax(ei_scores)]

# Pick winner by NN-predicted score
model.eval()
with torch.no_grad():
    nn_score_grad = model(torch.tensor(grad_candidate.astype(np.float32))).item()
    nn_score_gp   = model(torch.tensor(gp_candidate.astype(np.float32))).item()

selected = 'NN gradient ascent' if nn_score_grad >= nn_score_gp else 'GP EI'
next_x   = grad_candidate if nn_score_grad >= nn_score_gp else gp_candidate
portal_string = '-'.join([f'{v:.6f}' for v in next_x])

print('=' * 72)
print('  Acquisition Results')
print('=' * 72)
print(f'  Trust-region radius   : +-{trust_radius} per dimension')
grad_str = ', '.join(f'{v:.6f}' for v in grad_candidate)
gp_str   = ', '.join(f'{v:.6f}' for v in gp_candidate)
print(f'  NN gradient candidate : [{grad_str}]  (score: {nn_score_grad:.4f})')
print(f'  GP EI candidate      : [{gp_str}]  (score: {nn_score_gp:.4f})')
print(f'  Selected method       : {selected}')
print()
next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Next query point      : [{next_str}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

In [ ]:
# Cell 7: Input Gradient Sensitivity (PyTorch, Assignment 16.1 style)
# Compute |d(output)/d(x_i)| at best known point via backward().

x_sens = torch.tensor(best_X.copy().astype(np.float32), requires_grad=True)
model.eval()
score = model(x_sens)
score.backward()   # Assignment 16.1 backward
grads = x_sens.grad.abs().numpy()

best_x_str = ', '.join(f'{v:.6f}' for v in best_X)
print(f'Input gradient magnitudes at best X* = [{best_x_str}]:')
print()
max_g = grads.max() if grads.max() > 0 else 1.0
for i, g in enumerate(grads):
    bar = '#' * max(1, int(g * 40 / max_g))
    print(f'  x{i+1}: |grad| = {g:.4f}  {bar}')

most_sensitive = int(np.argmax(grads)) + 1
print(f'\n  Most sensitive dimension: x{most_sensitive}')
print(f'  Perturbing x{most_sensitive} has the largest predicted effect on output.')

In [ ]:
# Cell 8: SVM Classification Check
from sklearn.svm import SVC

threshold = np.percentile(Y, 70)
labels = (Y >= threshold).astype(int)

print(f'  Classification threshold (70th pct): {threshold:.6e}')
print(f'  Good samples: {labels.sum()} / {len(labels)}')
print()

if labels.sum() >= 2 and (labels == 0).sum() >= 2:
    svm_clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
    svm_clf.fit(X, labels)
    prob_best  = svm_clf.predict_proba(best_X.reshape(1, -1))[0, 1]
    prob_query = svm_clf.predict_proba(next_x.reshape(1, -1))[0, 1]
    print(f'  P(good) at best X*    : {prob_best:.3f}')
    print(f'  P(good) at next query : {prob_query:.3f}')
    if prob_query >= 0.5:
        print('  SVM endorses next query as likely good')
    elif prob_query >= 0.3:
        print('  SVM uncertain -- query near decision boundary')
    else:
        print('  SVM flags query as likely poor -- exploration risk accepted')
else:
    print('  Insufficient class balance for SVM.')

In [ ]:
print('=' * 72)
print('  SUMMARY -- Week 5 Bayesian Optimisation (PyTorch NN Surrogate)')
print('=' * 72)
print('  Function        : 4 -- Warehouse Placement (4D)')
print('  Objective       : Maximise')
print('  GP kernel       : RBF(length_scale=0.1, fixed)')
print(f'  NN architecture : MLP [{n_dims} -> 64 -> 64 -> 1], ReLU, Adam lr=1e-3, 1500 epochs')
print(f'  Acquisition     : NN gradient ascent (800 steps) vs GP EI (30k grid)')
print(f'  Selected method : {selected}')
print()
best_x_str = ', '.join(f'{v:.6f}' for v in best_X)
next_str   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best X*         : [{best_x_str}]')
print(f'  Best Y*         : {best_Y:.6e}')
print()
print(f'  Next query      : [{next_str}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

## Week 5 -- Reflection

**Function**: F4 -- Warehouse Placement (4D)

### Strategy note
W4 boundary query gave -48. Best is W1 -24.548. EI from W1 best; avoid corners.

### Query
- **Method**: PyTorch NN gradient ascent (800 steps, Adam lr=5e-3) vs GP EI (30k random grid)
- **Trust region**: ±0.1 per dimension from best known point
- **Winner selected by**: higher NN-predicted score
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 6
*(fill in after reflecting on result)*